### Setup

In [11]:
import pandas as pd
import os

from neo4j import GraphDatabase
from decimal import Decimal
from dotenv import load_dotenv

load_dotenv("../.env")

EDGES_DIR = "../data/edges/"

class Config:
    def __init__(self, mode="LOCAL", DATABASE='neo4j'):
        mode = mode.upper()

        if mode == "LOCAL":
            self.URI = os.getenv("NEO4J_URI_LOCAL")
            self.USER = os.getenv("NEO4J_USER_LOCAL")
            self.PASSWORD = os.getenv("NEO4J_PASSWORD_LOCAL")
        elif mode == "GROUP":
            self.URI = os.getenv("NEO4J_GROUP_URI")
            self.USER = os.getenv("NEO4J_GROUP_USER")
            self.PASSWORD = os.getenv("NEO4J_GROUP_PASSWORD")
        else:
            raise ValueError("Mode must be 'LOCAL' or 'GROUP'.")
        self.DATABASE = DATABASE


config = Config(mode="LOCAL",DATABASE = "dse203test")
driver = GraphDatabase.driver(config.URI, auth=(config.USER, config.PASSWORD))

with driver.session(database=config.DATABASE) as session:
    session.run("MATCH (n) RETURN n LIMIT 1")
    print(f"Connection successful")

def query_neo4j(cypher_query: str, parameters: dict = None):
    with driver.session(database=config.DATABASE) as session:
        result = session.run(cypher_query, parameters)
        return result.data()

Connection successful


In [12]:
def load_edge_json(json_path: str):
    """Load edge data supporting both array JSON and ndjson."""
    try:
        return pd.read_json(json_path)
    except ValueError:
        return pd.read_json(json_path, lines=True)

def get_relationship_count(label1, label2, relationship):
    result = query_neo4j(f"MATCH (n1:`{label1}`)-[r]-(n2:`{label2}`) RETURN DISTINCT TYPE(r) AS relationship, COUNT(*) as count")
    relationship_counts = {(i["relationship"]): i["count"] for i in result}
    return relationship_counts.get(relationship, 0)

def delete_relationships(label1, label2, relationship):
    existing_count = get_relationship_count(label1, label2, relationship)
    if existing_count > 0:
        query_neo4j(f"MATCH (a:`{label1}`)-[r:{relationship}]->(b:`{label2}`) DELETE r")
        new_count = get_relationship_count(label1, label2, relationship)
        assert new_count == 0, f"All {relationship} relationships between {label1} and {label2} were not deleted"
    print(f"Deleted {existing_count} `{relationship}` relationships between {label1} and {label2}.")

def create_relationships(label1, label2, relationship, data):
    query = f"""
    UNWIND $data AS row
    MATCH (a:`{label1}` {{id: row.id1}})
    MATCH (b:`{label2}` {{id: row.id2}})
    MERGE (a)-[r:{relationship}]->(b)
    SET r += coalesce(row.props, {{}})
    """
    query_neo4j(query, parameters={"data": data})
    new_count = get_relationship_count(label1, label2, relationship)
    print(f"Created {new_count} `{relationship}` relationships between {label1} and {label2}.")

def load_relationships(json_file):
    df = load_edge_json(EDGES_DIR + json_file)
    label1 = df.iloc[0]["entitytype1"]
    label2 = df.iloc[0]["entitytype2"]
    relationship = df.iloc[0]["predicate"]

    records = []
    for _, row in df.iterrows():
        rec = row.to_dict()
        id1 = int(rec.pop("entity1"))
        id2 = int(rec.pop("entity2"))
        rec.pop("entitytype1", None)
        rec.pop("entitytype2", None)
        rec.pop("predicate", None)
        props = {k: v for k, v in rec.items() if pd.notna(v)}
        records.append({"id1": id1, "id2": id2, "props": props})

    delete_relationships(label1, label2, relationship)
    create_relationships(label1, label2, relationship, records)

def view_relationships():
    result = query_neo4j("""
        MATCH (n1)-[r]-(n2)
        RETURN DISTINCT
            labels(n1) AS label1,
            type(r) AS relationship,
            labels(n2) AS label2,
            count(*) AS count
    """)
    
    data = [
        {
            "label1": record["label1"],
            "relationship": record["relationship"],
            "label2": record["label2"],
            "count": record["count"]
        }
        for record in result
    ]

    df = pd.DataFrame(data)
    return df


### Load Nodes From Json

In [13]:
relationship_files = os.listdir(EDGES_DIR)
relationship_files

['businesslocation_belongs_to_brand.json',
 'businesslocation_belongs_to_business.json',
 'businesslocation_contained_in_blockgroup.json',
 'businesslocation_contained_in_city.json',
 'businesslocation_contained_in_zipcode.json',
 'businesslocation_contained_in_zonelocation.json',
 'businesslocation_shared_region_businesslocation.json',
 'city_adjacent_to_city.json',
 'city_contained_in_county.json',
 'city_nearby_city.json',
 'community_adjacent_to_community.json',
 'community_contained_in_city.json',
 'community_nearby_community.json',
 'community_overlaps_blockgroup.json',
 'community_overlaps_zipcode.json',
 'county_adjacent_to_county.json',
 'county_contained_in_state.json',
 'zonelocation_belongs_to_zonetype.json']

In [14]:
for file in relationship_files:
    print(f"\nLoading relationships from file: {file}")
    load_relationships(file)


Loading relationships from file: businesslocation_belongs_to_brand.json
Deleted 0 `BELONGS_TO` relationships between BusinessLocation and Brand.
Created 39593 `BELONGS_TO` relationships between BusinessLocation and Brand.

Loading relationships from file: businesslocation_belongs_to_business.json
Deleted 0 `BELONGS_TO` relationships between BusinessLocation and Business.
Created 39593 `BELONGS_TO` relationships between BusinessLocation and Business.

Loading relationships from file: businesslocation_contained_in_blockgroup.json
Deleted 0 `CONTAINED_IN` relationships between BusinessLocation and BlockGroup.
Created 0 `CONTAINED_IN` relationships between BusinessLocation and BlockGroup.

Loading relationships from file: businesslocation_contained_in_city.json
Deleted 0 `CONTAINED_IN` relationships between BusinessLocation and City.
Created 38293 `CONTAINED_IN` relationships between BusinessLocation and City.

Loading relationships from file: businesslocation_contained_in_zipcode.json
De

## BusinessLocation Subsector links
Link existing BusinessLocation nodes to subsectors using derived CSV.


In [10]:
# Optional precheck
missing = fetch("""
LOAD CSV WITH HEADERS FROM 'file:///data/nodes/business_subsector.csv' AS row
WITH collect(toInteger(row.business_id)) AS ids
MATCH (b:BusinessLocation) WHERE b.id IN ids
RETURN size(ids) AS payload, count(b) AS matched, size(ids)-count(b) AS missing
""")
print(missing)

# Link BL -> Subsector
run("""
LOAD CSV WITH HEADERS FROM 'file:///data/nodes/business_subsector.csv' AS row
MATCH (b:BusinessLocation {id: toInteger(row.business_id)})
MATCH (sub:Subsector {name: row.subsector})
MERGE (b)-[:IN_SUBSECTOR]->(sub);
""")


NameError: name 'fetch' is not defined